# OneShop Data Lakehouse on AWS (emulated)

Same lakehouse as the chapter, built on AWS service names:

| Chapter component | AWS service here |
| --- | --- |
| MinIO | Amazon S3 |
| Iceberg REST fixture | AWS Glue Data Catalog (Iceberg REST endpoint) |
| `mc` CLI | AWS CLI |
| — | Amazon Athena (bonus query engine) |

Everything is served by Floci at `http://floci:4566`. No AWS account, no cost.

## 1. Confirm which account you are

In [ ]:
!aws sts get-caller-identity

## 2. The data lake bucket (Amazon S3)
Created by the init script at startup.

In [ ]:
!aws s3 ls

## 3. Create a database (AWS Glue Data Catalog)
In Iceberg terms this is a *namespace*; in Glue it is a *database*.

In [ ]:
%%sql
CREATE DATABASE IF NOT EXISTS glue.accounts

## 4. Create a partitioned Iceberg table

`truncate(1, last_name)` is Iceberg **hidden partitioning** — it partitions on
the first character of the last name without adding a column to the schema.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS glue.accounts.users (
    id INT,
    first_name STRING,
    last_name STRING,
    email STRING
)
USING iceberg
PARTITIONED BY (truncate(1, last_name))

### The table is now registered in Glue

Before inserting any rows, Iceberg has already written one metadata file.
Note that Spark created it, but **Glue** is what knows about it.

In [ ]:
!aws glue get-table --database-name accounts --name users \
    --query 'Table.Parameters' --output json

In [ ]:
!aws s3 ls s3://warehouse/accounts/users/metadata/ --recursive

## 5. Insert records

In [ ]:
%%sql
INSERT INTO glue.accounts.users (id, first_name, last_name, email) VALUES
(1, 'Juli', 'Arthars', 'jarthars0@mit.edu'),
(2, 'Matthiew', 'Hurley', 'mhurley1@narod.ru'),
(3, 'Lena', 'Westcarr', 'lwestcarr2@jimdo.com'),
(4, 'Reagen', 'Josifovitz', 'rjosifovitz3@hexun.com'),
(5, 'Ogden', 'Janecek', 'ojanecek4@yellowbook.com'),
(6, 'Calypso', 'McMurrugh', 'cmcmurrugh5@cam.ac.uk'),
(8, 'Jack', 'Matschoss', 'jmatschoss7@latimes.com'),
(9, 'Patty', 'Furnell', 'pfurnell8@yelp.com'),
(10, 'Lyndsie', 'Speeks', 'lspeeks9@skyrock.com')

### Inspect the partition layout in S3

This replaces the chapter's MinIO-console screenshot. You should see one
`last_name_trunc=<letter>` prefix per distinct first letter.

In [ ]:
!aws s3 ls s3://warehouse/accounts/users/data/

## 6. Query with Spark SQL

In [ ]:
%%sql
SELECT * FROM glue.accounts.users ORDER BY id

## 7. Query with PyIceberg

No JVM, no Spark context — a pure Python client against the same Glue catalog.

In [ ]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog('glue')
tbl = catalog.load_table('accounts.users')

# Partition pruning applies: we filter on the partitioned column.
sc = tbl.scan(row_filter="last_name LIKE 'M%'")
sc.to_arrow().to_pandas()

## 8. Query the same table with Amazon Athena

The chapter had no equivalent step. Because the table lives in the Glue Data
Catalog, Athena can read it with no extra setup — which is the real payoff of
using a catalog instead of a bare REST fixture.

In [ ]:
import boto3, time

athena = boto3.client('athena', endpoint_url='http://floci:4566')

q = athena.start_query_execution(
    QueryString='SELECT last_name, email FROM users ORDER BY last_name',
    QueryExecutionContext={'Database': 'accounts'},
    ResultConfiguration={'OutputLocation': 's3://athena-results/'},
)['QueryExecutionId']

while True:
    state = athena.get_query_execution(QueryExecutionId=q)['QueryExecution']['Status']['State']
    if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
        break
    time.sleep(1)

print('state:', state)
for row in athena.get_query_results(QueryExecutionId=q)['ResultSet']['Rows']:
    print([c.get('VarCharValue') for c in row['Data']])

## 9. Time travel

Iceberg keeps every snapshot, and the catalog hands any client the table's full snapshot log.

Spark's `glue.accounts.users.snapshots` metadata-table syntax does not work against Floci yet:
Iceberg first looks that name up as a table in namespace `accounts.users`, and Floci answers
the two-level namespace with an error instead of "not found", so Spark never falls back to the
metadata table. PyIceberg reads the same snapshot log directly.

In [ ]:
snapshots = catalog.load_table('accounts.users').inspect.snapshots()
snapshots.select(['snapshot_id', 'operation', 'committed_at']).to_pandas()

## Cleanup

```sql
DROP TABLE IF EXISTS glue.accounts.users
```